In [9]:
# Cell 1/7 — Imports + file paths (edit paths, then run)

import os                                  # file paths
import numpy as np                         # arrays
import pandas as pd                        # metadata table
import zarr                                # read Zarr store
import math
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm.auto import tqdm                 # progress bar

from sklearn.ensemble import RandomForestClassifier  # random forest (classification)
from sklearn.preprocessing import LabelEncoder       # convert hue strings -> ints
from sklearn.metrics import confusion_matrix, classification_report  # evaluation

# ---- EDIT THESE TWO PATHS ----
ZARR_PATH = "/Users/kate/Documents/retina-model/notebooks/export_retina_20260120_163636/dataset.zarr"   
META_CSV  = "/Users/kate/Documents/retina-model/notebooks/export_retina_20260120_163636/metadata.csv"   


In [10]:
# Cell 2/7 — Open the dataset + load metadata

root = zarr.open(ZARR_PATH, mode="r")      # open Zarr store (read-only)

imgs = root["imgs"]                        # (N,H,W,3) stimulus images
outs = root["outs"]                        # (N,K,H,W) raw responses (saved for completeness)
outs_fill = root["outs_fill"]              # (N,K,H,W) filled responses (main signal)

keys_order = list(root.attrs["keys_order"])# list of K response-map names (channel order)
meta = pd.read_csv(META_CSV)               # metadata table (one row per stimulus)

# Quick sanity checks
N = outs_fill.shape[0]                     # number of stimuli
K = outs_fill.shape[1]                     # number of response channels
H, W = outs_fill.shape[2], outs_fill.shape[3]
print("H,W:", H, W)
print("N stimuli:", N)
print("K channels:", K)
print("keys_order:", keys_order)


H,W: 1000 1000
N stimuli: 350
K channels: 8
keys_order: ['l_on', 'l_off', 'm_on', 'm_off', 'l_h2_on', 'm_h2_on', 'l_h2_off', 'm_h2_off']


In [11]:
# Cell 3/7 — Define the 3 channel-sets + helper functions

# Channel subsets you asked for
keys_all8 = ["l_on","l_off","m_on","m_off","l_h2_on","m_h2_on","l_h2_off","m_h2_off"]
keys_classic4 = ["l_on","l_off","m_on","m_off"]
keys_h2_4 = ["l_h2_on","m_h2_on","l_h2_off","m_h2_off"]

# Map key name -> channel index in the saved arrays
key_to_idx = {k: i for i, k in enumerate(keys_order)}

#def normalize_map(A):
#    """Normalize one 2D map so intensity scale is reduced (robust, per-map)."""
#    A = A.astype(np.float32, copy=False)                 # ensure float32
#    A = A - np.median(A)                                 # remove offset (centers distribution)
#    s = np.percentile(np.abs(A), 99)                     # robust scale (ignores extreme outliers)
#    if s > 0:                                            # avoid divide-by-zero
#        A = A / s                                        # scale to comparable magnitude
#    return A                                             # return normalized map

def normalize_map(A, eps=1e-12):
    """Z-score one 2D map: subtract mean, divide by std (per-map)."""
    A = A.astype(np.float32, copy=False)          # ensure float32
    mu = float(A.mean())                           # map mean
    sd = float(A.std())                            # map standard deviation
    if sd < eps:                                   # avoid divide-by-zero / tiny std
        return (A - mu)                             # if nearly constant, just center
    return (A - mu) / sd                            # z-scored map

#def pool_to_64(A, crop=256):
#    """
#    Center-crop to (crop x crop), then average-pool to (64 x 64).
#    With crop=256, pooling is 4x4 blocks -> 64.
#    """
#    H, W = A.shape                                       # current shape (should be 1000x1000)
#    cy, cx = H // 2, W // 2                              # center coordinates
#    half = crop // 2                                     # half crop size
#    A = A[cy-half:cy+half, cx-half:cx+half]              # center crop -> (crop,crop)
#    A = A.reshape(64, crop//64, 64, crop//64).mean(axis=(1, 3))  # avg-pool -> (64,64)
#    return A.astype(np.float32)                          # return float32

def build_X_for_keys(keys_subset, desc="building X"):
    """
    Build feature matrix X for a chosen subset of channels.
    Each stimulus becomes one feature vector by:
      (1) selecting channels
      (2) per-map normalize
      (3) flatten and concatenate across channels
    """
    idxs = [key_to_idx[k] for k in keys_subset]           # channel indices for this subset
    X = np.zeros((N, len(idxs) * H * W), dtype=np.float32)  # preallocate features

    for i in tqdm(range(N), desc=desc):                   # loop over stimuli
        feat_parts = []                                   # collect per-channel features
        for ch in idxs:                                   # loop over selected channels
            A = np.asarray(outs_fill[i, ch, :, :], dtype=np.float32)                    # read one filled map (H,W)
#            A = normalize_map(A)                          # normalize
#            A = pool_to_64(A, crop=256)                   # downsample to (64,64)
            feat_parts.append(A.ravel())                  # flatten and store
        X[i] = np.concatenate(feat_parts, axis=0)         # concatenate all channels
    return X                                              # (N, n_features)


In [12]:
# Cell 4/7 — Build X matrices and y labels (hue classification target)

# Build features for each input option
X_all8 = build_X_for_keys(keys_all8,     desc="X_all8 (8 channels)")
X_cl4  = build_X_for_keys(keys_classic4, desc="X_classic4 (4 channels)")
X_h2   = build_X_for_keys(keys_h2_4,     desc="X_h2 (4 channels)")

# Labels: hue strings -> integer classes
le = LabelEncoder()                                      # encoder for hue labels
y_str = meta["hue"].astype(str).values                   # hue column as strings
y = le.fit_transform(y_str)                              # numeric labels 0..C-1

print("Class labels:", list(le.classes_))                 # show hue class names
print("X_all8 shape:", X_all8.shape)
print("X_cl4  shape:", X_cl4.shape)
print("X_h2   shape:", X_h2.shape)


X_all8 (8 channels):   0%|          | 0/350 [00:00<?, ?it/s]

X_classic4 (4 channels):   0%|          | 0/350 [00:00<?, ?it/s]

X_h2 (4 channels):   0%|          | 0/350 [00:00<?, ?it/s]

Class labels: ['blue', 'green', 'red', 'white', 'yellow']
X_all8 shape: (350, 8000000)
X_cl4  shape: (350, 4000000)
X_h2   shape: (350, 4000000)


In [13]:
# Cell 5/7 — random 75/25 holdout split (BALANCED train by hue, seed=0)

np.random.seed(0)                                        # tutorial-style fixed seed (unchanged)

all_idx = np.arange(N)                                   # all stimulus indices (unchanged)
n_train = int(0.75 * N)                                  # target train size (unchanged)

# --- NEW: build a balanced train set (same # per hue) ---
classes = np.unique(y)                                   # CHANGED: get numeric class ids present
idx_by_class = {c: all_idx[y == c] for c in classes}      # CHANGED: indices for each hue class

# CHANGED: per-class train count is limited by the smallest class
n_train_per_class = min(len(idxs) for idxs in idx_by_class.values())  # CHANGED: max balanced per-class
n_train_per_class = int(np.floor(0.75 * n_train_per_class))           # CHANGED: take 75% of each class

train_parts = []                                         # CHANGED: collect sampled train indices
for c in classes:                                        # CHANGED: loop over classes
    idxs = idx_by_class[c]                               # CHANGED: indices for this class
    pick = np.random.choice(idxs, size=n_train_per_class, replace=False)  # CHANGED: sample per class
    train_parts.append(pick)                             # CHANGED: store picks

train_idx = np.concatenate(train_parts)                  # CHANGED: combine per-class samples
test_idx = np.setdiff1d(all_idx, train_idx)              # CHANGED: everything else is test (includes extra W/Y)

# (Optional) sort for nicer reproducibility/printing
#train_idx = np.sort(train_idx)                           # sort indices (unchanged)
#test_idx = np.sort(test_idx)                             # sort indices (unchanged)

print("Train N:", len(train_idx), " Test N:", len(test_idx))  # unchanged
print("Train counts by hue:", np.bincount(y[train_idx]))       # CHANGED: show balance check
print("Test  counts by hue:", np.bincount(y[test_idx]))        # CHANGED: show test distribution


Train N: 185  Test N: 165
Train counts by hue: [37 37 37 37 37]
Test  counts by hue: [13 13 13 63 63]


In [14]:
# Cell 6/7 — Train 3 RandomForestClassifier models 

MAX_DEPTH = 5                                             # fixed (as requested)
N_EST = 200                                               # number of trees (reasonable default)

def fit_rf(X, name):
    """Fit one RF classifier on the tutorial split."""
    rf = RandomForestClassifier(
        n_estimators=N_EST,                               # number of trees
        max_depth=MAX_DEPTH,                              # fixed depth
        random_state=0,                                   # fixed randomness (tutorial style)
        n_jobs=-1                                         # use all CPU cores
    )
    rf.fit(X[train_idx], y[train_idx])                    # train
    return rf                                             # return trained model

rf_all8 = fit_rf(X_all8, "all8")
print("ALL8  train:", rf_all8.score(X_all8[train_idx], y[train_idx]))
print("ALL8  test :", rf_all8.score(X_all8[test_idx],  y[test_idx]))

rf_cl4 = fit_rf(X_cl4, "classic4")
print("CL4   train:", rf_cl4.score(X_cl4[train_idx], y[train_idx]))
print("CL4   test :", rf_cl4.score(X_cl4[test_idx],  y[test_idx]))

rf_h2 = fit_rf(X_h2, "h2_4")
print("H2    train:", rf_h2.score(X_h2[train_idx], y[train_idx]))
print("H2    test :", rf_h2.score(X_h2[test_idx],  y[test_idx]))


ALL8  train: 1.0
ALL8  test : 0.9757575757575757
CL4   train: 0.9351351351351351
CL4   test : 0.6666666666666666
H2    train: 1.0
H2    test : 0.9696969696969697


In [15]:
# Cell 7/7 — Evaluate (+ confusion matrices)

def evaluate(rf, X, title):
    """Print tutorial-style .score plus useful classification diagnostics."""
    acc = rf.score(X[test_idx], y[test_idx])              # .score = accuracy for classifier
    print("\n" + "="*70)
    print(title)
    print(f"Test accuracy (.score): {acc:.4f}")

    y_pred = rf.predict(X[test_idx])                      # predicted classes
    cm = confusion_matrix(y[test_idx], y_pred)            # confusion matrix
    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)

    # Optional: more detailed report (precision/recall/F1 per hue)
    print("\nClassification report:")
    print(classification_report(y[test_idx], y_pred, target_names=le.classes_))

evaluate(rf_all8, X_all8, "RF hue prediction — ALL 8 channels")
evaluate(rf_cl4,  X_cl4,  "RF hue prediction — CLASSICAL 4 channels")
evaluate(rf_h2,   X_h2,   "RF hue prediction — H2+ 4 channels")



RF hue prediction — ALL 8 channels
Test accuracy (.score): 0.9758
Confusion matrix (rows=true, cols=pred):
[[12  0  0  1  0]
 [ 0 12  0  1  0]
 [ 0  0 13  0  0]
 [ 0  0  0 63  0]
 [ 0  0  0  2 61]]

Classification report:
              precision    recall  f1-score   support

        blue       1.00      0.92      0.96        13
       green       1.00      0.92      0.96        13
         red       1.00      1.00      1.00        13
       white       0.94      1.00      0.97        63
      yellow       1.00      0.97      0.98        63

    accuracy                           0.98       165
   macro avg       0.99      0.96      0.97       165
weighted avg       0.98      0.98      0.98       165


RF hue prediction — CLASSICAL 4 channels
Test accuracy (.score): 0.6667
Confusion matrix (rows=true, cols=pred):
[[13  0  0  0  0]
 [ 2 10  0  1  0]
 [ 0  0 13  0  0]
 [ 1 15  0 38  9]
 [ 1  7  0 19 36]]

Classification report:
              precision    recall  f1-score   support

    

In [16]:
# Cell 8 — Export misclassified test stimuli (RGB) for ALL8 / CLASSIC4 / H2+4
# Helper: build a df of misclassified test stimuli for one model
def build_df_wrong(meta, test_idx, y_true, y_pred, le):
    """Return a dataframe listing which test stimuli were misclassified."""
    wrong = (y_pred != y_true)                                    # boolean: wrong predictions
    wrong_stim_id = test_idx[wrong]                               # stim_ids of wrong cases (these are global ids)

    df_wrong = meta.iloc[wrong_stim_id].copy()                    # pull metadata rows for those stim_ids
    df_wrong["stim_id"] = wrong_stim_id                           # record stim_id explicitly
    df_wrong["true_hue"] = le.inverse_transform(y_true[wrong])     # decode numeric -> string
    df_wrong["pred_hue"] = le.inverse_transform(y_pred[wrong])     # decode numeric -> string
    return df_wrong.reset_index(drop=True)                        # clean index for nicer printing

# ----------------------------
# Helper: save an "album" (contact sheet + individual PNGs) for misclassified cases
def save_misclass_album(df_wrong, imgs, outdir, model_tag, max_cols=5, dpi=200):
    """
    Save:
      1) a CSV summary of errors
      2) individual PNGs of each misclassified stimulus
      3) one contact-sheet PNG with captions
    """
    os.makedirs(outdir, exist_ok=True)                            # create folder if needed

    # If no errors, just print and exit
    if len(df_wrong) == 0:
        print(f"[{model_tag}] No misclassified test images to save.")
        return

    # ---- 1) Save the table of misclassified cases
    csv_path = os.path.join(outdir, f"errors_{model_tag}.csv")     # filename for summary table
    df_wrong.to_csv(csv_path, index=False)                         # write table to disk

    # ---- 2) Save each misclassified stimulus as its own PNG
    img_dir = os.path.join(outdir, f"images_{model_tag}")          # subfolder for individual images
    os.makedirs(img_dir, exist_ok=True)                            # create subfolder

    for _, row in df_wrong.iterrows():                             # loop over each misclassified case
        sid = int(row["stim_id"])                                  # stimulus id (row in imgs)
        true_h = str(row["true_hue"])                               # true label
        pred_h = str(row["pred_hue"])                               # predicted label
        inten  = float(row.get("intensity", np.nan))               # intensity if present

        # Build a filename that won't overwrite and contains key info
        if np.isfinite(inten):
            fname = f"stim{sid:04d}_true-{true_h}_pred-{pred_h}_I{inten:.2f}.png"
        else:
            fname = f"stim{sid:04d}_true-{true_h}_pred-{pred_h}.png"

        fpath = os.path.join(img_dir, fname)                       # full output path
        plt.imsave(fpath, imgs[sid])                               # save the RGB image (expects values in [0,1])

    # ---- 3) Save a single "contact sheet" album image
    n = len(df_wrong)                                              # number of wrong images
    cols = min(int(max_cols), n)                                   # number of columns in the grid
    rows = int(math.ceil(n / cols))                                # number of rows in the grid

    fig, axs = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))  # create grid of axes
    axs = np.array(axs).reshape(-1)                                # flatten axes to 1D list for easy looping

    for ax in axs:                                                 # default: turn all axes off
        ax.axis("off")

    for j, (_, row) in enumerate(df_wrong.iterrows()):             # loop over errors for placement in grid
        sid = int(row["stim_id"])                                  # stimulus id
        true_h = str(row["true_hue"])                               # true hue
        pred_h = str(row["pred_hue"])                               # predicted hue
        inten  = float(row.get("intensity", np.nan))               # intensity

        axs[j].imshow(imgs[sid])                                   # show the stimulus
        if np.isfinite(inten):
            axs[j].set_title(f"id {sid} | true={true_h} | pred={pred_h} | I={inten:.2f}", fontsize=9)
        else:
            axs[j].set_title(f"id {sid} | true={true_h} | pred={pred_h}", fontsize=9)
        axs[j].axis("off")                                         # no ticks

    fig.suptitle(f"Misclassified test stimuli — {model_tag} (n={n})", fontsize=12)
    fig.tight_layout()

    album_path = os.path.join(outdir, f"album_{model_tag}.png")     # output path for contact sheet
    fig.savefig(album_path, dpi=dpi, bbox_inches="tight")          # save the album image
    plt.close(fig)                                                 # free memory

    print(f"[{model_tag}] Saved:")
    print("  ", csv_path)
    print("  ", img_dir)
    print("  ", album_path)

# ----------------------------
# Build y_true once (for the shared test split)
y_true = y[test_idx]                                               # ground-truth labels for test set

# Predict for each model
y_pred_all8 = rf_all8.predict(X_all8[test_idx])                    # predictions for ALL8
y_pred_cl4  = rf_cl4.predict(X_cl4[test_idx])                      # predictions for CLASSIC4
y_pred_h2   = rf_h2.predict(X_h2[test_idx])                        # predictions for H2+4

# Build df_wrong for each model
df_wrong_all8 = build_df_wrong(meta, test_idx, y_true, y_pred_all8, le)
df_wrong_cl4  = build_df_wrong(meta, test_idx, y_true, y_pred_cl4,  le)
df_wrong_h2   = build_df_wrong(meta, test_idx, y_true, y_pred_h2,   le)

# Output directory (timestamped)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")                      # timestamp string
OUTDIR = f"misclassified_albums_{ts}"                              # folder name
os.makedirs(OUTDIR, exist_ok=True)                                 # create folder

# Save albums
save_misclass_album(df_wrong_all8, imgs, OUTDIR, model_tag="ALL8")
save_misclass_album(df_wrong_cl4,  imgs, OUTDIR, model_tag="CLASSIC4")
save_misclass_album(df_wrong_h2,   imgs, OUTDIR, model_tag="H2_4")


[ALL8] Saved:
   misclassified_albums_20260120_170955/errors_ALL8.csv
   misclassified_albums_20260120_170955/images_ALL8
   misclassified_albums_20260120_170955/album_ALL8.png
[CLASSIC4] Saved:
   misclassified_albums_20260120_170955/errors_CLASSIC4.csv
   misclassified_albums_20260120_170955/images_CLASSIC4
   misclassified_albums_20260120_170955/album_CLASSIC4.png
[H2_4] Saved:
   misclassified_albums_20260120_170955/errors_H2_4.csv
   misclassified_albums_20260120_170955/images_H2_4
   misclassified_albums_20260120_170955/album_H2_4.png
